# Ariadne — Planning Agent Playground

Interactively drive the federated **planning agent** (`utils/planner/`): read which graphs are live in Docker, then plan analyst questions over them.

The planner *routes* — it never queries. For a question it emits a machine-readable **plan**: target sources, sub-questions (each with traversal limits), cross-source join keys, and explicitly **unsupported** parts. A fully deterministic **gate** validates every plan before it ships (*the model drafts, code grades*).

## Prerequisites
- **Graph stack running:** `uv run main.py --docker start` — CAPEC in Gremlin, ATT&CK in Neo4j.
- **Source cards built:** `uv run main.py --download && uv run main.py --build all`.
- *(Optional)* a local OpenAI-compatible LLM for the `use_llm=True` path (default `http://localhost:9090/v1`, model `default-model`).

## Launch (from the repo root)
```bash
uv run --with jupyterlab jupyter lab notebooks/planner_playground.ipynb
```
`--with` provides Jupyter for the session, so you don't have to add it to the project dependencies.

> Tip: the equivalent one-shot CLI is `uv run main.py --plan "<question>"` (add `--plan-llm` to draft with the model).

In [ ]:
# --- setup: make `utils.planner` importable no matter where the notebook lives ---
import sys, os, json, dataclasses
from pathlib import Path

_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()), _here)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.config import load_env
load_env()  # load the central .env (NEO4J_*/GREMLIN_*/PLANNER_* knobs); real env vars still win

from utils.planner import PlanningAgent, PlannerConfig, load_source_cards
from utils.planner.discovery import get_live_sources
from utils.planner.join_keys import resolve_join_key
from utils.planner.evaluation import run_corpus, aggregate

print("repo root:", ROOT)

## 1. Which graphs are live right now?

The planner runs a **content probe** per backend (connect → non-trivial node count → expected labels present), each bounded by a short timeout. Anything unreachable, unauthorized, or empty is reported *down-with-reason* — and the planner plans around it.

In [ ]:
config = PlannerConfig.from_env()
cards = load_source_cards(config.data_dir)

if not cards:
    print("No source cards found.\nBuild them first:  uv run main.py --download && uv run main.py --build all")
else:
    view = get_live_sources(cards, config)
    for ds in sorted(view.sources):
        s = view.sources[ds]
        flag = "LIVE" if s.live else "DOWN"
        detail = f"{s.node_count} nodes, labels={list(s.observed_labels)}" if s.live else s.reason
        print(f"[{flag}] {ds:6s} ({s.backend:7s}) @ {s.endpoint}")
        print(f"        {detail}")
    print("\nlive:", view.live_datasets(), "| down:", view.down_datasets())

## 2. Inspect the source cards

The planner routes over the **introspected** structural fields (`node_labels`, `relationship_types`) plus enriched safety limits, and only joins across sources on the single **declared canonical key**.

In [ ]:
for ds, card in cards.items():
    print(f"=== {ds}  ({card.runtime_backend}) ===")
    print("  node_labels:       ", sorted(card.node_labels))
    print("  relationship_types:", sorted(card.relationship_types))
    print("  limits:            ", card.limits())
    print()

if {"attack", "capec"} <= set(cards):
    key = resolve_join_key(cards["attack"], cards["capec"])
    print("canonical join key:", key.name if key else None)
    if key:
        print(" ", key.description)

## 3. Plan a question

`plan_question(q)` builds an agent (which re-probes liveness), drafts a plan, runs it through the deterministic gate, and prints a readable summary. Pass `use_llm=True` to draft with the local model instead of the deterministic router.

In [ ]:
def make_agent(use_llm=False):
    cfg = dataclasses.replace(PlannerConfig.from_env(), llm_enabled=use_llm)
    return PlanningAgent(cfg)

# Reuse agents across calls (each .plan() re-probes within a short TTL cache).
agent = make_agent(use_llm=False)
llm_agent = None  # created lazily on first use_llm=True call

def show_plan(plan):
    print("  routed to:", sorted(plan.target_sources()) or "(none)")
    for sq in plan.sub_questions:
        print(f"    - [{sq.source}] {sq.entities}  limits={sq.limits.to_dict()}")
    for j in plan.joins:
        print(f"    x join '{j.key}' on {j.sources}")
    for u in plan.unsupported:
        print(f"    ! unsupported: {u.description}  ({u.reason[:80]})")
    drafter = plan.trace[-1]["drafter"] if plan.trace else "?"
    print(f"  partial={plan.partial} | drafter={drafter} | iterations={len(plan.trace)}")

def plan_question(q, use_llm=False):
    global llm_agent
    if use_llm:
        llm_agent = llm_agent or make_agent(use_llm=True)
        a = llm_agent
    else:
        a = agent
    print("Q:", q)
    plan = a.plan(q)
    show_plan(plan)
    return plan

_ = plan_question(
    "Which CAPEC patterns related to credential theft map to ATT&CK techniques, "
    "and what mitigations appear for those techniques?"
)

## 4. Run a batch of queries

Edit `QUESTIONS` and re-run. Watch how the planner marks parts **unsupported** when a needed source is down, and refuses **out-of-scope** requests instead of fabricating a path.

In [ ]:
QUESTIONS = [
    "Which adversary groups use which techniques?",
    "Which CWE weaknesses do CAPEC attack patterns exploit?",
    "What execution steps make up a CAPEC attack pattern?",
    "List all Kubernetes pods and their CPU usage.",   # out of scope -> unsupported
]
for q in QUESTIONS:
    plan_question(q)
    print()

## 5. Draft with the local LLM

Uses the OpenAI-compatible endpoint at `PLANNER_LLM_BASE_URL` (default `http://localhost:9090/v1`, model `PLANNER_LLM_MODEL`). If the endpoint is unreachable or returns something unparseable, the agent transparently falls back to the deterministic drafter — so this cell always returns a valid plan.

In [ ]:
_ = plan_question(
    "Which CAPEC patterns related to credential theft map to ATT&CK techniques, "
    "and what mitigations appear for those techniques?",
    use_llm=True,
)

## 6. The full machine-readable plan (with trace)

This JSON is exactly what the downstream specialist agents would consume — sub-questions with limits, declared joins, unsupported parts, and the per-iteration gate trace.

In [ ]:
plan = plan_question("Which adversary groups use which techniques, and what mitigations apply?")
print()
print(json.dumps(plan.to_dict(), indent=2))

## 7. Score the planner on the expected-plan corpus

Plan-level metrics only — source selection, entity recall, unnecessary source calls, schema adherence, and honesty (correct `unsupported`/`partial`) — decoupled from any downstream answer. Uses the deterministic drafter and a fixed live view, so it runs even if the backends are down.

In [ ]:
_cards = cards if cards else load_source_cards("data")
if not _cards:
    print("Build source cards first:  uv run main.py --build all")
else:
    results = run_corpus(_cards)
    scores = [s for _c, _p, s in results]
    for s in scores:
        flag = "PASS" if s.passed else "FAIL"
        print(f"  [{flag}] {s.name:32s} src_recall={s.source_recall:.2f} "
              f"ent_recall={s.entity_recall:.2f} unnec={s.unnecessary_source_calls} "
              f"schema={'ok' if s.schema_adherent else 'BAD'}")
    print("\naggregate:", json.dumps(aggregate(scores), indent=2))

## Troubleshooting

- **No source cards** -> `uv run main.py --download && uv run main.py --build all`.
- **A source shows DOWN** -> start the stack with `uv run main.py --docker start`. For Neo4j, make sure `.env` has the right `NEO4J_PASSWORD` (the planner honestly marks an unauthenticated Neo4j as down rather than guessing).
- **The LLM path falls back to deterministic** -> check the model server at `PLANNER_LLM_BASE_URL` and set `PLANNER_LLM_MODEL` to match. The deterministic path always works offline.
- **Tune behavior via env** (or `.env`): `PLANNER_PROBE_TIMEOUT_S`, `PLANNER_CACHE_TTL_S`, `PLANNER_MAX_ITERATIONS`, `PLANNER_MIN_LIVE_NODES`, `PLANNER_LLM_MAX_TOKENS`.
- **Full reference:** [`docs/planner_agent.md`](../docs/planner_agent.md).